# Baseline Strategy Comparison

Use this notebook to compare the latest available run from each preprocessing strategy and rank the combined candidates from best to worst.

In [1]:
from pathlib import Path
import pandas as pd

baseline_root = Path('.')
strategies = ['zscore', 'client_zscore', 'magnitude_features', 'magnitude_only', 'robust_clip']
ranking_columns = ['pr_auc', 'miss_rate', 'far', 'balanced_accuracy', 'f1']
ranking_ascending = [False, True, True, False, False]

strategy_runs = []
for strategy in strategies:
    results_root = baseline_root / strategy / 'results'
    run_dirs = sorted([path for path in results_root.glob('run_*') if path.is_dir()]) if results_root.exists() else []
    latest_run = run_dirs[-1] if run_dirs else None
    strategy_runs.append({
        'strategy': strategy,
        'latest_run': None if latest_run is None else latest_run.name,
        'run_path': None if latest_run is None else str(latest_run),
        'available': latest_run is not None,
    })

strategy_runs_df = pd.DataFrame(strategy_runs)
strategy_runs_df

,strategy,latest_run,run_path,available
0,zscore,run_20260329_195901,zscore\results\run_20260329_195901,True
1,client_zscore,run_20260329_203944,client_zscore\results\run_20260329_203944,True
2,magnitude_features,run_20260329_205220,magnitude_features\results\run_20260329_205220,True
3,magnitude_only,run_20260329_210333,magnitude_only\results\run_20260329_210333,True
4,robust_clip,run_20260329_211240,robust_clip\results\run_20260329_211240,True


## Load Latest Results

This cell loads the latest run available for each preprocessing strategy. Strategies without results yet are skipped automatically.

In [2]:
global_frames = []
dataset_frames = []

for row in strategy_runs:
    if not row['available']:
        continue

    run_path = Path(row['run_path'])
    metrics_global = pd.read_csv(run_path / 'metrics_global.csv')
    metrics_by_dataset = pd.read_csv(run_path / 'metrics_by_dataset.csv')

    metrics_global['preprocessing_strategy'] = row['strategy']
    metrics_by_dataset['preprocessing_strategy'] = row['strategy']

    metrics_global['model_candidate'] = metrics_global['selected_candidate']
    metrics_by_dataset['model_candidate'] = metrics_by_dataset['selected_candidate']

    metrics_global['selected_candidate_label'] = (
        metrics_global['preprocessing_strategy'] + ' + ' + metrics_global['model'] + ' (' + metrics_global['selected_candidate'] + ')'
    )
    metrics_by_dataset['selected_candidate_label'] = (
        metrics_by_dataset['preprocessing_strategy'] + ' + ' + metrics_by_dataset['model'] + ' (' + metrics_by_dataset['selected_candidate'] + ')'
    )

    global_frames.append(metrics_global)
    dataset_frames.append(metrics_by_dataset)

all_global = pd.concat(global_frames, ignore_index=True) if global_frames else pd.DataFrame()
all_by_dataset = pd.concat(dataset_frames, ignore_index=True) if dataset_frames else pd.DataFrame()

print(f"Loaded {len(all_global)} global rows across {len(global_frames)} strategy runs.")
print(f"Loaded {len(all_by_dataset)} dataset rows across {len(dataset_frames)} strategy runs.")

Loaded 20 global rows across 5 strategy runs.
Loaded 60 dataset rows across 5 strategy runs.


## Final Comparison Table

This table ranks every strategy-model combination from best to worst using the fall-detection priority: higher PR-AUC, lower miss rate, lower FAR, higher balanced accuracy, and higher F1.

In [3]:
if all_global.empty:
    final_comparison = pd.DataFrame(columns=['selected_candidate'])
else:
    final_comparison = (
        all_global[
            [
                'selected_candidate_label',
                'preprocessing_strategy',
                'model',
                'model_candidate',
                'accuracy',
                'balanced_accuracy',
                'specificity',
                'precision',
                'recall',
                'f1',
                'roc_auc',
                'pr_auc',
                'far',
                'miss_rate',
                'tn',
                'fp',
                'fn',
                'tp',
            ]
        ]
        .rename(columns={'selected_candidate_label': 'selected_candidate'})
        .sort_values(ranking_columns, ascending=ranking_ascending)
        .reset_index(drop=True)
    )

final_comparison

,selected_candidate,preprocessing_strategy,model,model_candidate,accuracy,balanced_accuracy,specificity,precision,recall,f1,roc_auc,pr_auc,far,miss_rate,tn,fp,fn,tp
0,magnitude_features + neural_network (mlp_wide),magnitude_features,neural_network,mlp_wide,0.976705,0.975939,0.978473,0.960362,0.973404,0.966839,0.997350,0.995453,0.021527,0.026596,16136,355,235,8601
1,magnitude_features + xgboost (xgb_deep),magnitude_features,xgboost,xgb_deep,0.976981,0.974916,0.981748,0.966008,0.968085,0.967045,0.996980,0.995068,0.018252,0.031915,16190,301,282,8554
2,zscore + neural_network (mlp_wide),zscore,neural_network,mlp_wide,0.961977,0.962239,0.961373,0.930360,0.963105,0.946449,0.993069,0.988267,0.038627,0.036895,15854,637,326,8510
3,zscore + xgboost (xgb_deep),zscore,xgboost,xgb_deep,0.960674,0.956589,0.970105,0.944142,0.943074,0.943608,0.991598,0.986304,0.029895,0.056926,15998,493,503,8333
4,magnitude_features + knn (knn_k5),magnitude_features,knn,knn_k5,0.958621,0.957140,0.962040,0.930752,0.952241,0.941374,0.984040,0.966047,0.037960,0.047759,15865,626,422,8414
5,robust_clip + xgboost (xgb_deep),robust_clip,xgboost,xgb_deep,0.923244,0.916079,0.939785,0.888150,0.892372,0.890256,0.976157,0.959321,0.060215,0.107628,15498,993,951,7885
6,zscore + knn (knn_k5),zscore,knn,knn_k5,0.940143,0.937172,0.947001,0.903617,0.927343,0.915326,0.974156,0.945756,0.052999,0.072657,15617,874,642,8194
7,client_zscore + neural_network (mlp_wide),client_zscore,neural_network,mlp_wide,0.844158,0.855979,0.816870,0.723671,0.895088,0.800304,0.940563,0.903232,0.183130,0.104912,13471,3020,927,7909
8,client_zscore + xgboost (xgb_deep),client_zscore,xgboost,xgb_deep,0.860742,0.830495,0.930568,0.849322,0.730421,0.785397,0.937983,0.900128,0.069432,0.269579,15346,1145,2382,6454
9,robust_clip + knn (knn_k11),robust_clip,knn,knn_k11,0.869625,0.861797,0.887696,0.799524,0.835899,0.817307,0.938474,0.875990,0.112304,0.164101,14639,1852,1450,7386


## Results by Dataset

This view compares all strategy-model combinations separately on KFall, SisFall, and UpFall.

In [4]:
if all_by_dataset.empty:
    dataset_comparison = pd.DataFrame(columns=['dataset', 'selected_candidate'])
else:
    dataset_comparison = (
        all_by_dataset[
            [
                'dataset',
                'selected_candidate_label',
                'preprocessing_strategy',
                'model',
                'model_candidate',
                'accuracy',
                'balanced_accuracy',
                'specificity',
                'precision',
                'recall',
                'f1',
                'roc_auc',
                'pr_auc',
                'far',
                'miss_rate',
            ]
        ]
        .rename(columns={'selected_candidate_label': 'selected_candidate'})
        .sort_values(['dataset', *ranking_columns], ascending=[True, *ranking_ascending])
        .reset_index(drop=True)
    )

dataset_comparison

,dataset,selected_candidate,preprocessing_strategy,model,model_candidate,accuracy,balanced_accuracy,specificity,precision,recall,f1,roc_auc,pr_auc,far,miss_rate
0,KFall,magnitude_features + xgboost (xgb_deep),magnitude_features,xgboost,xgb_deep,0.989763,0.989362,0.992732,0.990723,0.985992,0.988352,0.999540,0.999408,0.007268,0.014008
1,KFall,magnitude_features + neural_network (mlp_wide),magnitude_features,neural_network,mlp_wide,0.988641,0.988901,0.986717,0.983260,0.991086,0.987157,0.999389,0.999219,0.013283,0.008914
2,KFall,zscore + neural_network (mlp_wide),zscore,neural_network,mlp_wide,0.979386,0.979750,0.976692,0.970755,0.982808,0.976744,0.997853,0.997296,0.023308,0.017192
3,KFall,zscore + xgboost (xgb_deep),zscore,xgboost,xgb_deep,0.969149,0.967689,0.979950,0.974034,0.955428,0.964642,0.996569,0.995611,0.020050,0.044572
4,KFall,robust_clip + xgboost (xgb_deep),robust_clip,xgboost,xgb_deep,0.948815,0.944099,0.983709,0.977632,0.904489,0.939639,0.994713,0.992760,0.016291,0.095511
5,KFall,magnitude_features + knn (knn_k5),magnitude_features,knn,knn_k5,0.978685,0.979191,0.974937,0.968642,0.983445,0.975987,0.995790,0.991806,0.025063,0.016555
6,KFall,zscore + knn (knn_k5),zscore,knn,knn_k5,0.960875,0.960397,0.964411,0.954863,0.956383,0.955623,0.990370,0.984516,0.035589,0.043617
7,KFall,robust_clip + knn (knn_k11),robust_clip,knn,knn_k11,0.909129,0.900201,0.975188,0.963211,0.825215,0.888889,0.981694,0.974567,0.024812,0.174785
8,KFall,robust_clip + neural_network (mlp_wide),robust_clip,neural_network,mlp_wide,0.921750,0.916764,0.958647,0.943357,0.874881,0.907830,0.981509,0.969662,0.041353,0.125119
9,KFall,client_zscore + neural_network (mlp_wide),client_zscore,neural_network,mlp_wide,0.808442,0.821506,0.711779,0.717791,0.931232,0.810698,0.935922,0.928330,0.288221,0.068768


## Interpretation of Results

This section summarizes the main conclusions from the strategy-model comparison using the fall-detection ranking.

In [5]:
from IPython.display import Markdown, display

if final_comparison.empty or dataset_comparison.empty:
    display(Markdown('No comparison results are available yet.'))
else:
    best_global = final_comparison.iloc[0]
    runner_up = final_comparison.iloc[1] if len(final_comparison) > 1 else None

    best_per_dataset = dataset_comparison.groupby('dataset', as_index=False).first()
    easiest_dataset = best_per_dataset.sort_values(['pr_auc', 'miss_rate'], ascending=[False, True]).iloc[0]
    hardest_dataset = best_per_dataset.sort_values(['pr_auc', 'miss_rate'], ascending=[True, False]).iloc[0]

    strategy_summary = (
        final_comparison.groupby('preprocessing_strategy', as_index=False)
        .agg(
            mean_pr_auc=('pr_auc', 'mean'),
            mean_miss_rate=('miss_rate', 'mean'),
            mean_far=('far', 'mean'),
            best_rank_pr_auc=('pr_auc', 'max'),
        )
        .sort_values(['mean_pr_auc', 'mean_miss_rate', 'mean_far'], ascending=[False, True, True])
        .reset_index(drop=True)
    )

best_strategy = strategy_summary.iloc[0]
lowest_far_row = final_comparison.sort_values('far', ascending=True).iloc[0]
lowest_miss_rate_row = final_comparison.sort_values('miss_rate', ascending=True).iloc[0]

summary_lines = [
    '### Key Takeaways',
    '',
    (
        f"- **Best overall combination:** `{best_global['selected_candidate']}` ranks first with PR-AUC = **{best_global['pr_auc']:.4f}**, miss rate = **{best_global['miss_rate']:.4f}**, and FAR = **{best_global['far']:.4f}**. Under the current ranking, this is the strongest balance between detecting falls and avoiding unnecessary false alarms."
    ),
    (
        ' '.join([
            f"- **Closest competitor:** `{runner_up['selected_candidate']}` is the next best option with PR-AUC = **{runner_up['pr_auc']:.4f}**"
            f" and miss rate = **{runner_up['miss_rate']:.4f}**." if runner_up is not None else '- **Closest competitor:** only one candidate is available.'
        ])
        if runner_up is not None else '- **Closest competitor:** only one candidate is available.'
    ),
    (
        f"- **Best preprocessing strategy on average:** `{best_strategy['preprocessing_strategy']}` has the strongest mean performance across its models, with average PR-AUC = **{best_strategy['mean_pr_auc']:.4f}**, average miss rate = **{best_strategy['mean_miss_rate']:.4f}**, and average FAR = **{best_strategy['mean_far']:.4f}**."
    ),
    (
        f"- **Dataset difficulty:** `{easiest_dataset['dataset']}` appears easiest in the current setup (top PR-AUC = **{easiest_dataset['pr_auc']:.4f}**), while `{hardest_dataset['dataset']}` is the most challenging (top PR-AUC = **{hardest_dataset['pr_auc']:.4f}**, miss rate = **{hardest_dataset['miss_rate']:.4f}**)."
    ),
    (
        f"- **Error trade-offs:** `{lowest_miss_rate_row['selected_candidate']}` achieves the lowest miss rate (**{lowest_miss_rate_row['miss_rate']:.4f}**), which is especially important when missing a fall is costly. `{lowest_far_row['selected_candidate']}` achieves the lowest FAR (**{lowest_far_row['far']:.4f}**), which is preferable when minimizing false positives is the priority."
    ),
]

display(Markdown('\n'.join(summary_lines)))

### Key Takeaways

- **Best overall combination:** `magnitude_features + neural_network (mlp_wide)` ranks first with PR-AUC = **0.9955**, miss rate = **0.0266**, and FAR = **0.0215**. Under the current ranking, this is the strongest balance between detecting falls and avoiding unnecessary false alarms.
- **Closest competitor:** `magnitude_features + xgboost (xgb_deep)` is the next best option with PR-AUC = **0.9951** and miss rate = **0.0319**.
- **Best preprocessing strategy on average:** `magnitude_features` has the strongest mean performance across its models, with average PR-AUC = **0.9373**, average miss rate = **0.0418**, and average FAR = **0.0702**.
- **Dataset difficulty:** `KFall` appears easiest in the current setup (top PR-AUC = **0.9994**), while `UpFall` is the most challenging (top PR-AUC = **0.9153**, miss rate = **0.1305**).
- **Error trade-offs:** `magnitude_features + neural_network (mlp_wide)` achieves the lowest miss rate (**0.0266**), which is especially important when missing a fall is costly. `magnitude_features + xgboost (xgb_deep)` achieves the lowest FAR (**0.0183**), which is preferable when minimizing false positives is the priority.